# 08 — PSMA-PET-CT-Lesions: study inventory and 68Ga cohort split

The PSMA-PET-CT-Lesions collection (autoPET, LMU Munich) is the source of the pre-therapy PSMA PET/CT used as network input and, for the 68Ga-PSMA-11 studies, for the PBPK fitting. This notebook builds a table with one row per study, as the reference for every later selection, and a patient-level train/test split of the 68Ga studies.

For each study the table holds:
- **metadata**: subject, study date, age, scanner, tracer (18F or 68Ga), CT contrast agent;
- **geometry**: matrix, voxel spacing, axial (z) extent and in-plane field of view, read from the NIfTI headers; check that CT, PET and label share the same grid;
- **lesions**: number and volume of the connected components of the lesion mask (`labelsTr`), lesion SUVmax and SUVmean, total lesion uptake.

**Steps**
1. Metadata and file matching.
2. Per-study measures (cached on disk).
3. Overview of the collection by tracer.
4. Patient-level train/test split of the 68Ga studies.
5. Saving.

**About this notebook.** Inventory of the PSMA-PET-CT-Lesions studies and patient-level 68Ga split.
- **Inputs:** `$LU177_DATASET/PSMA-PET-CT-Lesions_v3/`; `src/psma_exclusions.py`.
- **Outputs:** `data/processed/psma/psma_inventory.csv`, `psma_lesions.csv`, `psma_68ga_split.*`, `psma_study_measures.csv`.
- **Run time:** < 1 min with the cached measures; about 20 min from scratch.
- **Environment:** .venv.

In [ ]:
import json
import multiprocessing as mp
from concurrent.futures import ProcessPoolExecutor
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import nibabel as nib
import numpy as np
import pandas as pd
from scipy import ndimage

WS = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "src" / "paths.py").exists())
sys.path.insert(0, str(WS / "src"))
from paths import DATASET
DATA = DATASET / "PSMA-PET-CT-Lesions_v3"
OUT = WS / "data" / "processed" / "psma"
OUT.mkdir(parents=True, exist_ok=True)

CONNECTIVITY = np.ones((3, 3, 3), dtype=bool)
MIN_LESION_ML = 1.0
WORKERS = 8
RECOMPUTE = False

TEST_FRACTION = 0.2
SEED = 20261031

COLORS = {"18F": "#2a78d6", "68Ga": "#eb6834"}
TRACERS = ["18F", "68Ga"]
plt.rcParams.update({"figure.facecolor": "#fcfcfb", "axes.facecolor": "#fcfcfb", "axes.edgecolor": "#52514e",
                     "axes.labelcolor": "#0b0b0b", "xtick.color": "#52514e", "ytick.color": "#52514e",
                     "axes.spines.top": False, "axes.spines.right": False, "axes.grid": True,
                     "grid.color": "#e4e3df", "grid.linewidth": 0.6, "lines.linewidth": 2})

## 1. Metadata and file matching

`psma_metadata.csv` identifies a study by `Subject ID` (upper-case `PSMA_` prefix) and `Study Date`; the files are named `psma_<subject>_<date>`, with `_0000` = CT resampled to the PET grid and `_0001` = PET in SUV (the channel names in `dataset.json` both say CT, which is wrong). The study identifier used everywhere below is the file stem.

In [ ]:
meta = pd.read_csv(DATA / "psma_metadata.csv").rename(columns={
    "Subject ID": "subject", "Study Date": "study_date", "manufacturer_model_name": "scanner",
    "pet_radionuclide": "tracer", "ct_contrast_agent": "ct_contrast"})
meta["subject"] = meta["subject"].str.lower()
meta["study_id"] = meta["subject"] + "_" + meta["study_date"]
meta["ct_contrast"] = meta["ct_contrast"].map({"yes": True, "no": False})

labels = {p.name.removesuffix(".nii.gz") for p in (DATA / "labelsTr").glob("*.nii.gz")}
images = {p.name.removesuffix(".nii.gz")[:-5] for p in (DATA / "imagesTr").glob("*.nii.gz")}
complete = {s for s in labels if (DATA / "imagesTr" / f"{s}_0000.nii.gz").exists()
            and (DATA / "imagesTr" / f"{s}_0001.nii.gz").exists()}

print(f"metadata rows {len(meta)}, unique study ids {meta.study_id.nunique()}")
print(f"label files {len(labels)}, image stems {len(images)}, studies with CT + PET + label {len(complete)}")
print("in metadata but without files:", sorted(set(meta.study_id) - complete))
print("files without metadata:", sorted((labels | images) - set(meta.study_id)))

## 2. Per-study measures

Definitions:
- **Geometry** from the PET header: matrix, spacing (mm), axial extent $n_z \Delta z$ and in-plane field of view $n_x \Delta x$; orientation codes; flags that the CT and the label have the same shape and affine as the PET.
- **Lesions** = connected components of the binary label with 26-connectivity (voxels touching by a face, edge or corner belong to the same lesion), in voxel units of the PET grid. The count is sensitive to this choice and to small fragments of the manual segmentation: with 6-connectivity the same masks give up to ~20% more components in the cases checked. For this reason the count of components of at least 1 mL is also stored, and the per-lesion table is saved so that any other threshold can be applied later.
- **SUV**: lesion SUVmax = maximum SUV over all lesion voxels; lesion SUVmean = mean over all lesion voxels (volume-weighted over lesions); total lesion uptake TLU = SUVmean × total lesion volume (mL); body SUVmax over the whole PET volume, as a sanity check of the SUV scaling.

Studies without lesions have zero lesions and volume, and missing SUV values. The measures take about one second per study; they are computed in parallel and cached in `data/processed/psma/`.

In [ ]:
def measure(study_id):
    pet_img = nib.load(DATA / "imagesTr" / f"{study_id}_0001.nii.gz")
    ct_img = nib.load(DATA / "imagesTr" / f"{study_id}_0000.nii.gz")
    lab_img = nib.load(DATA / "labelsTr" / f"{study_id}.nii.gz")
    nx, ny, nz = pet_img.shape
    dx, dy, dz = (float(v) for v in pet_img.header.get_zooms()[:3])
    voxel_ml = dx * dy * dz / 1000.0
    suv = np.asarray(pet_img.dataobj, dtype=np.float32)
    mask = np.asarray(lab_img.dataobj) > 0
    cc, n = ndimage.label(mask, CONNECTIVITY)
    row = {"study_id": study_id, "nx": nx, "ny": ny, "nz": nz, "dx_mm": dx, "dy_mm": dy, "dz_mm": dz,
           "z_extent_mm": nz * dz, "fov_xy_mm": nx * dx, "orientation": "".join(nib.aff2axcodes(pet_img.affine)),
           "ct_same_grid": ct_img.shape == pet_img.shape and np.allclose(ct_img.affine, pet_img.affine, atol=1e-3),
           "label_same_grid": lab_img.shape == pet_img.shape and np.allclose(lab_img.affine, pet_img.affine, atol=1e-3),
           "label_values": ",".join(str(v) for v in np.unique(np.asarray(lab_img.dataobj))),
           "body_suvmax": float(suv.max()), "n_lesions": n}
    lesions = []
    if n:
        idx = np.arange(1, n + 1)
        nvox = np.bincount(cc.ravel(), minlength=n + 1)[1:]
        sums = ndimage.sum_labels(suv, cc, idx)
        maxs = ndimage.maximum(suv, cc, idx)
        centroids = ndimage.center_of_mass(mask, cc, idx)
        for k in range(n):
            z_mm = float(pet_img.affine[2] @ np.array([*centroids[k], 1.0]))
            lesions.append({"study_id": study_id, "lesion": k + 1, "n_voxels": int(nvox[k]),
                            "volume_ml": float(nvox[k] * voxel_ml), "suvmax": float(maxs[k]),
                            "suvmean": float(sums[k] / nvox[k]), "centroid_z_mm": z_mm})
        tlv = float(nvox.sum() * voxel_ml)
        suvmean = float(sums.sum() / nvox.sum())
        row.update({"n_lesions_ge_1ml": int((nvox * voxel_ml >= MIN_LESION_ML).sum()), "tlv_ml": tlv,
                    "largest_lesion_ml": float(nvox.max() * voxel_ml), "median_lesion_ml": float(np.median(nvox) * voxel_ml),
                    "lesion_suvmax": float(maxs.max()), "lesion_suvmean": suvmean, "tlu": suvmean * tlv})
    else:
        row.update({"n_lesions_ge_1ml": 0, "tlv_ml": 0.0, "largest_lesion_ml": 0.0, "median_lesion_ml": np.nan,
                    "lesion_suvmax": np.nan, "lesion_suvmean": np.nan, "tlu": 0.0})
    return row, lesions


study_csv, lesion_csv = OUT / "psma_study_measures.csv", OUT / "psma_lesions.csv"
if RECOMPUTE or not study_csv.exists():
    with ProcessPoolExecutor(WORKERS, mp_context=mp.get_context("fork")) as pool:
        results = list(pool.map(measure, sorted(complete), chunksize=4))
    measures = pd.DataFrame([r for r, _ in results])
    lesions = pd.DataFrame([l for _, ls in results for l in ls])
    measures.to_csv(study_csv, index=False)
    lesions.to_csv(lesion_csv, index=False)
else:
    measures, lesions = pd.read_csv(study_csv), pd.read_csv(lesion_csv)

inventory = meta.merge(measures, on="study_id", how="inner", validate="one_to_one")
inventory["n_studies_subject"] = inventory.groupby("subject").study_id.transform("size")
print(f"{len(inventory)} studies, {inventory.subject.nunique()} subjects, {len(lesions)} lesions")
print("CT on PET grid:", inventory.ct_same_grid.all(), "| label on PET grid:", inventory.label_same_grid.all())
print("orientations:", inventory.orientation.value_counts().to_dict(), "| label values:", inventory.label_values.value_counts().to_dict())
inventory.head()

## 3. Overview by tracer

Collection-level numbers per tracer. Spacing and axial extent depend on the scanner and protocol; lesion burden is very skewed (a few studies with hundreds of components), so medians and quartiles are reported.

In [ ]:
def q(s):
    s = s.dropna()
    return f"{s.median():.3g} [{s.quantile(0.25):.3g}-{s.quantile(0.75):.3g}]" if len(s) else "-"


def overview(df):
    return pd.Series({
        "studies": len(df), "subjects": df.subject.nunique(),
        "subjects with >1 study": int((df.groupby("subject").size() > 1).sum()),
        "contrast CT": f"{df.ct_contrast.sum()} ({df.ct_contrast.mean():.0%})",
        "scanners": ", ".join(f"{k} {v}" for k, v in df.scanner.value_counts().items()),
        "age, median [IQR]": q(df.age),
        "spacing x/y/z (mm)": ", ".join(f"{a:.2f}/{b:.2f}/{c:.2f}: {n}" for (a, b, c), n
                                         in df[["dx_mm", "dy_mm", "dz_mm"]].round(2).value_counts().items()),
        "z extent (mm), median [IQR]": q(df.z_extent_mm),
        "z extent (mm), min-max": f"{df.z_extent_mm.min():.0f}-{df.z_extent_mm.max():.0f}",
        "lesion-free studies": int((df.n_lesions == 0).sum()),
        "lesions per study, median [IQR]": q(df.n_lesions),
        "lesions >= 1 mL per study, median [IQR]": q(df.n_lesions_ge_1ml),
        "TLV (mL), median [IQR]": q(df.tlv_ml),
        "lesion SUVmax, median [IQR]": q(df.lesion_suvmax),
        "lesion SUVmean, median [IQR]": q(df.lesion_suvmean),
        "body SUVmax, median [IQR]": q(df.body_suvmax),
    })


pd.set_option("display.max_colwidth", 120)
pd.DataFrame({t: overview(inventory[inventory.tracer == t]) for t in TRACERS} | {"all": overview(inventory)})

In [ ]:
lesions_t = lesions.merge(inventory[["study_id", "tracer"]], on="study_id")
fig, axes = plt.subplots(2, 2, figsize=(11, 7.5))
panels = [
    (axes[0, 0], lambda d: d.z_extent_mm, np.arange(600, inventory.z_extent_mm.max() + 50, 40), False, "axial extent (mm)", "studies"),
    (axes[0, 1], lambda d: d.n_lesions + 1, np.unique(np.round(np.logspace(0, np.log10(inventory.n_lesions.max() + 2), 30))) - 0.5 + 1e-9, True, "lesions per study + 1", "studies"),
    (axes[1, 0], lambda d: d.tlv_ml[d.tlv_ml > 0], np.logspace(-1.5, np.log10(inventory.tlv_ml.max() * 1.1), 30), True, "total lesion volume (mL), lesion-positive studies", "studies"),
    (axes[1, 1], lambda d: d.suvmax, np.logspace(0, np.log10(lesions.suvmax.max() * 1.1), 30), True, "lesion SUVmax (per lesion)", "lesions"),
]
for ax, get, bins, logx, xlabel, ylabel in panels:
    for t in TRACERS:
        src = lesions_t if ylabel == "lesions" else inventory
        ax.hist(get(src[src.tracer == t]), bins=bins, histtype="step", color=COLORS[t], linewidth=2,
                label=f"{t} (n = {len(get(src[src.tracer == t]))})")
    if logx:
        ax.set_xscale("log")
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    ax.legend(frameon=False)
fig.suptitle("PSMA-PET-CT-Lesions: geometry and lesion burden by tracer", x=0.01, ha="left")
fig.tight_layout()
fig.savefig(OUT / "psma_inventory_overview.png", dpi=150)
plt.show()

## 4. Patient-level train/test split of the 68Ga studies

The PBPK cohort uses only the 68Ga-PSMA-11 studies (the 18F agent is not specified). The split is done on **subjects**, never on studies, because several subjects have two or three 68Ga studies: studies of the same patient on both sides would leak anatomy and uptake pattern into the test set.

Rules:
- test fraction 20% of the 68Ga subjects, fixed seed;
- stratified on subject-level features, so that train and test have the same mix: lesion burden (lesion-free, or tertile of the median total lesion volume over the subject's 68Ga studies) and whether the subject has more than one 68Ga study (lesion-free subjects form a single stratum, they are only 8). A first version stratified only on lesion-free vs lesion-positive gave a test set with half the median TLV of the training set, which is why the burden tertiles were added;
- within each stratum the subjects are sorted, shuffled with the seed and the first `round(0.2 n)` go to test.

Some 68Ga subjects also have 18F studies. These are not part of the PBPK cohort, but the inventory marks them with the subject's split (`subject_split_68ga`): if 18F studies are later used for pre-training the network, the ones of test subjects must be excluded.

Studies in `psma_pet_exclusions.csv` with action `exclude` (written by `src/psma_exclusions.py`: wrong SUV from a wrong injection time or weight) get no `split_68ga`. They are removed after the assignment, which is still computed on all 68Ga studies, so the split of every other study does not depend on the exclusions.

In [ ]:
ga = inventory[inventory.tracer == "68Ga"]
subj = ga.groupby("subject").agg(n_studies=("study_id", "size"), any_lesion=("n_lesions", lambda s: bool((s > 0).any())),
                                 tlv_ml=("tlv_ml", "median"))
subj["multi_study"] = subj.n_studies > 1
subj["has_18F"] = subj.index.isin(inventory.loc[inventory.tracer == "18F", "subject"])
pos = subj.any_lesion
subj["burden"] = "lesion-free"
subj.loc[pos, "burden"] = pd.qcut(subj.loc[pos, "tlv_ml"], 3, labels=["TLV T1", "TLV T2", "TLV T3"]).astype(str)
subj["stratum"] = subj.burden + np.where(pos & subj.multi_study, " / multi", np.where(pos, " / single", ""))

rng = np.random.default_rng(SEED)
subj["split"] = "train"
for key, group in subj.groupby("stratum"):
    names = np.array(sorted(group.index))
    rng.shuffle(names)
    subj.loc[names[:int(round(TEST_FRACTION * len(names)))], "split"] = "test"

excluded = pd.read_csv(OUT / "psma_pet_exclusions.csv").query("action == 'exclude'").study_id
inventory["split_68ga"] = inventory.subject.map(subj.split).where((inventory.tracer == "68Ga") & ~inventory.study_id.isin(excluded))
inventory["subject_split_68ga"] = inventory.subject.map(subj.split)

print("TLV tertile edges (mL):", np.round(subj.loc[pos, "tlv_ml"].quantile([1 / 3, 2 / 3]).values, 1))
print(pd.crosstab(subj.stratum, subj.split, margins=True))

In [ ]:
def balance(df, s):
    return pd.Series({
        "subjects": len(s), "studies": len(df), "subjects with >1 study": int(s.multi_study.sum()),
        "lesion-free studies": int((df.n_lesions == 0).sum()), "contrast CT": int(df.ct_contrast.sum()),
        "subjects also with 18F studies": int(s.has_18F.sum()),
        "age, median [IQR]": q(df.age), "z extent (mm), median [IQR]": q(df.z_extent_mm),
        "lesions per study, median [IQR]": q(df.n_lesions), "TLV (mL), median [IQR]": q(df.tlv_ml),
        "lesion SUVmax, median [IQR]": q(df.lesion_suvmax),
        "scanners": ", ".join(f"{k} {v}" for k, v in df.scanner.value_counts().items()),
    })


pd.DataFrame({sp: balance(ga[(ga.subject.map(subj.split) == sp) & ~ga.study_id.isin(excluded)], subj[subj.split == sp]) for sp in ["train", "test"]})

The collection also ships `splits_final.json`, the 5-fold split of the nnU-Net baseline. It is not used here; the check below only reports whether its folds keep the studies of one subject together, which matters if a pre-trained nnU-Net model from those folds is ever reused.

In [ ]:
folds = json.loads((DATA / "splits_final.json").read_text())
for k, f in enumerate(folds):
    train_s = {"_".join(s.split("_")[:2]) for s in f["train"]}
    val_s = {"_".join(s.split("_")[:2]) for s in f["val"]}
    print(f"fold {k}: {len(f['train'])}/{len(f['val'])} studies, subjects on both sides: {len(train_s & val_s)}")

## 5. Saving

Outputs in `data/processed/psma/`:
- `psma_inventory.csv`: one row per study (metadata, geometry, lesion summary, `split_68ga` for the 68Ga studies, `subject_split_68ga` for every study of a 68Ga subject);
- `psma_lesions.csv`: one row per connected component (volume, SUVmax, SUVmean, centroid z);
- `psma_study_measures.csv`: cache of the image measures;
- `psma_68ga_split.csv` (one row per 68Ga study) and `psma_68ga_split.json` (subject lists, seed and rules);
- `psma_inventory_overview.png`.

In [ ]:
inventory.to_csv(OUT / "psma_inventory.csv", index=False)
split_rows = inventory.loc[inventory.split_68ga.notna(), ["subject", "study_id", "study_date", "split_68ga"]].rename(
    columns={"split_68ga": "split"}).sort_values(["split", "subject", "study_date"])
split_rows.to_csv(OUT / "psma_68ga_split.csv", index=False)
(OUT / "psma_68ga_split.json").write_text(json.dumps({
    "description": "Patient-level train/test split of the 68Ga-PSMA-11 studies of PSMA-PET-CT-Lesions_v3 (PBPK cohort)",
    "seed": SEED, "test_fraction": TEST_FRACTION, "stratified_on": ["lesion-free or tertile of median TLV over 68Ga studies", "more than one 68Ga study"],
    "excluded_studies": sorted(set(excluded) & set(ga.study_id)),
    "train_subjects": sorted(subj.index[subj.split == "train"]), "test_subjects": sorted(subj.index[subj.split == "test"]),
    "train_studies": sorted(split_rows.study_id[split_rows.split == "train"]),
    "test_studies": sorted(split_rows.study_id[split_rows.split == "test"]),
}, indent=2))
assert not set(subj.index[subj.split == "train"]) & set(subj.index[subj.split == "test"])
print(split_rows.split.value_counts().to_dict(), subj.split.value_counts().to_dict())